In [1]:
import os

# Update you config directory as needed
os.environ["PSYNC_CONFIG_DIR"] = "../../config"


%load_ext autoreload
%autoreload 2

# Tidal

The following sections describe how to use the Tidal service with `plistsync`.

## Getting started

If you have not done so already, please make sure to set the `tidal` configuration in your `/config/config.yaml` file:

```yaml
# ./config/config.yaml
tidal:
    enabled: true
    client_id: your_tidal_client_id
    client_secret: your_tidal_client_secret (optional)
```

You can check if this is set up correctly by running:

```python
from plistsync.config import Config
Config().tidal.__dict__
```

To use Tidal, you need to have a Tidal account. To authenticate and authorize `plistsync` to access your Tidal account, run `plistsync tidal auth` and follow the prompts.

```{typer} plistsync.services.tidal.authenticate:tidal_cli
---
prog: plistsync tidal auth
---
```

## Playlists

### Loading a specific playlist by ID

You can load a specific Tidal playlist by its ID using the `TidalPlaylistCollection` class. You may find the id of the playlist in the URL when you open it in your browser. Or by using the tidal sharing feature. E.g. the playlist with the URL `https://tidal.com/playlist/a17f77e4-f640-4cb6-bf55-59b6fbb21bda` has the ID `a17f77e4-f640-4cb6-bf55-59b6fbb21bda`.


In [2]:
from plistsync.services.tidal import TidalPlaylistCollection

litt_dnb = await TidalPlaylistCollection.from_id("a17f77e4-f640-4cb6-bf55-59b6fbb21bda")

# Alternatively, you may use asyncio.run to run the method in a synchronous context:
import asyncio

litt_dnb = asyncio.run(
    TidalPlaylistCollection.from_id("a17f77e4-f640-4cb6-bf55-59b6fbb21bda")
)

INFO:plistsync:Tidal rate limit exceeded: Waiting 4 seconds


In [6]:
tracks = list(litt_dnb)
for track in tracks[:10]:
    print(f"{track.primary_artist} - {track.name}")

Dryman - No Respect
Exile - Take Me Away (Come Hard Mix)
Friction - Find Our Way
Mat Zo - Games
Jaguar Skills - Off My Rocker
Grima x Azza - DnB Artform
Cid Rim - Repeat
TC - Deep (feat. Jakes)
Plump DJs - Rocket Soul
Camo & Krooked - If I Could (feat. Joe Killington)


Sometimes tracks or albums are removed from tidal and will not be available in the playlist anymore. These tracks will be skipped when loading the playlist and you will see a warning message. Sadly it is not possible to retrieve these tracks in the v2 api see [this issue](https://github.com/orgs/tidal-music/discussions/231).


### Loading all playlist of the current user

When you want to load all playlists of the currently authenticated user, you can use the `TidalPlaylistCollection.playlists` property. Depending on how many playlists you have, this might take a while.

In [7]:
from plistsync.services.tidal import TidalLibraryCollection

user_playlists = TidalLibraryCollection().playlists

INFO:plistsync:Tidal rate limit exceeded: Waiting 4 seconds                   
INFO:plistsync:Tidal rate limit exceeded: Waiting 4 seconds                   
INFO:plistsync:Tidal rate limit exceeded: Waiting 4 seconds                    
INFO:plistsync:Tidal rate limit exceeded: Waiting 4 seconds                    


### Accessing playlist properties

When fetching playlists from Tidal using the methods above, you get a `TidalPlaylistCollection` object. You can access various properties of these objects, such as:
- `name`: The name of the playlist.
- `description`: The description of the playlist.
- `__iter__`: You can iterate over the tracks in the playlist.
- `data`: The raw data of the playlist as returned by the Tidal API.

In [9]:
litt_dnb.name

'Litt DnB 19'

In [10]:
litt_dnb.id

'a17f77e4-f640-4cb6-bf55-59b6fbb21bda'

In [11]:
# Use as iterable to get all tracks
tracks = [t for t in litt_dnb]

In [12]:
# Get raw data
raw_data = litt_dnb.data
# And included data as lookup dictionary
lookup = litt_dnb.data_lookup

## Tracks

### Retrieving tracks

You may want to retrieve tracks manually using their Tidal IDs or ISRCs. You can do this using the `TidalCollection.find_by_global_ids` and `TidalCollection.find_many_by_global_ids` methods.


In [ ]:
from plistsync.services.tidal import TidalLibraryCollection

# By id
track = TidalLibraryCollection().find_by_global_ids({"tidal_id": "451416503"})
track

Track[Born On Road > Flowers, 8764913632057]

In [ ]:
# By isrc
track = TidalLibraryCollection().find_by_global_ids({"isrc": "GB2LD2510402"})
track

Track[Born On Road > Flowers, 8764913634556]

### Accessing track properties

When we fetch tracks from Tidal using the methods above, we get a `TidalTrack` object or `TidalPlaylistTrack` object. You can access various properties of these objects, such as:
- `name`: The name of the track.
- `artists`: A list of artist names associated with the track.
- `data`: The raw data of the track as returned by the Tidal API.

In [ ]:
track.name

'Flowers'

In [11]:
track.artists

['Born On Road', 'Brodie']

In [12]:
# You can access the raw data returned by the Tidal API as well, e.g. the relationships:
track.data["relationships"]

{'albums': {'data': [{'id': '451416501', 'type': 'albums'}],
  'links': {'self': '/tracks/451416503/relationships/albums?countryCode=DE'}},
 'trackStatistics': {'links': {'self': '/tracks/451416503/relationships/trackStatistics?countryCode=DE'}},
 'artists': {'data': [{'id': '21732726', 'type': 'artists'},
   {'id': '4642490', 'type': 'artists'}],
  'links': {'self': '/tracks/451416503/relationships/artists?countryCode=DE'}},
 'genres': {'links': {'self': '/tracks/451416503/relationships/genres?countryCode=DE'}},
 'similarTracks': {'links': {'self': '/tracks/451416503/relationships/similarTracks?countryCode=DE'}},
 'owners': {'links': {'self': '/tracks/451416503/relationships/owners?countryCode=DE'}},
 'lyrics': {'links': {'self': '/tracks/451416503/relationships/lyrics?countryCode=DE'}},
 'sourceFile': {'links': {'self': '/tracks/451416503/relationships/sourceFile?countryCode=DE'}},
 'providers': {'links': {'self': '/tracks/451416503/relationships/providers?countryCode=DE'}},
 'radio'